### Import the necessary libraries

In [1]:
import pandas as pd
from collections import Counter


### Read the saved CSV file

In [2]:
df_full = pd.read_csv("all_data_with_liwc.csv")
print(df_full.head())   # See first 5 rows
print(df_full.columns)  # See column names

                                 model culture                 topic  \
0  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
1  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
2  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
3  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
4  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   

   temperature  generation_number       character_pair        stance  \
0          0.9                  4  Mother and Daughter       Neutral   
1          0.7                  4  Father and Daughter  Disagreement   
2          0.3                  1       Father and Son  Disagreement   
3          0.5                  5  Father and Daughter  Disagreement   
4          0.7                  3       Son and Mother  Disagreement   

                                                text  affect  posemo  ...  \
0  My dear mother, you know how the world is chan...     

### Filtering llama8B

In [4]:
# ---- Step 2: Filter the DataFrame only by model ----
filtered_df_llama_8b_model = df_full[
    df_full["model"].str.contains("meta-llama_Llama-3.1-8B-Instruct", na=False)
]

# ---- Step 3: Display the filtered data and its shape ----
if filtered_df_llama_8b_model.empty:
    print("⚠️ No data available for the specified model filter.")
else:
    print("✅ Filtered Data (by model only):")
    print(filtered_df_llama_8b_model.head())  # Display first few rows
    print("\nShape of filtered data:", filtered_df_llama_8b_model.shape)

✅ Filtered Data (by model only):
                                    model culture                 topic  \
6000  meta-llama_Llama-3.1-8B-Instruct_UK      UK  Interracial Marriage   
6001  meta-llama_Llama-3.1-8B-Instruct_UK      UK  Interracial Marriage   
6002  meta-llama_Llama-3.1-8B-Instruct_UK      UK  Interracial Marriage   
6003  meta-llama_Llama-3.1-8B-Instruct_UK      UK  Interracial Marriage   
6004  meta-llama_Llama-3.1-8B-Instruct_UK      UK  Interracial Marriage   

      temperature  generation_number       character_pair        stance  \
6000          0.9                  4  Mother and Daughter       Neutral   
6001          0.7                  4  Father and Daughter  Disagreement   
6002          0.3                  1       Father and Son  Disagreement   
6003          0.5                  5  Father and Daughter  Disagreement   
6004          0.7                  3       Son and Mother  Disagreement   

                                                   text  affect  

In [9]:
output_path = "all_data_with_liwc.csv"

print(f"\n[✅] LIWC-enriched data saved to: {output_path}")



[✅] LIWC-enriched data saved to: all_data_with_liwc.csv


In [11]:
print(f"\n[✅] LIWC-enriched data saved to: {output_path}")
print(df_full.head(10))  # Preview the enriched DataFrame

import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr, spearmanr
from sklearn.neural_network import MLPRegressor

# ---- Load ELFeN features ----
df1 = filtered_df_llama_8b_model
#df1['culture']

#Replace 'UK' with 1 and 'Iran' with 0 in the 'culture' column
df1['culture'] = df1['culture'].replace({'UK': 1, 'Iran': 0})

# Verify the changes
#print(df1['culture'].head())

# keep only numeric feature columns
df1= df1.select_dtypes(include=["float64", "int64", "uint16", "uint32", "float32"])
df1.shape

# ---- Separate features and label ----
feat_cols = [c for c in df1.columns if c!="culture"]
lab_cols = [c for c in df1.columns if c=="culture"]

data = df1[feat_cols]
labels = df1[lab_cols]

print(data.head())
print(data.shape)
print()
print(labels.head())
print(labels.shape)

# ---- Scale features ----
scaler = StandardScaler()
data[data.columns] = scaler.fit_transform(data[data.columns])

# Add constant for statsmodels
data = sm.add_constant(data)
print(data.head())

# ---- Statistical approach ----
print("Statistical approach: ")
model = sm.OLS(labels["culture"], data)
res = model.fit()
print(res.summary())

# Save summary
with open("filtered_df_llama_8b_model_Output_LIWC_Stats.txt", "w") as text_file:
    text_file.write(str(res.summary()))
    
# ---- Machine learning approach ----
print("Machine learning approach:")
X_train, X_test, y_train, y_test = train_test_split(
    data, labels["culture"], test_size=0.20, random_state=42, shuffle=True
)

model = LinearRegression(fit_intercept=False)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Pearson correlation on test set:", pearsonr(list(preds), y_test))
print("Spearman correlation on test set:", spearmanr(list(preds), y_test))

weights = pd.DataFrame()
weights["feature"] = data.columns
weights["weight"] = model.coef_
print(weights)
print()
print()


# ---- Non-linear model ----
print("Machine learning with non-linear models (might take a little while):")
model = MLPRegressor(max_iter=10000)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Pearson correlation on test set:", pearsonr(list(preds), y_test))
print("Spearman correlation on test set:", spearmanr(list(preds), y_test))


from sklearn.linear_model import Ridge
model = Ridge(alpha=900)  # Try with different alpha values
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Pearson correlation on test set:", pearsonr(list(preds), y_test))
print("Spearman correlation on test set:", spearmanr(list(preds), y_test))


import pandas as pd
import re

# Path to your .txt file
file_path = "/home/shekhar/Desktop/code/persona/filtered_df_llama_8b_model_Output_LIWC_Stats.txt"

# Read the .txt file line by line
with open(file_path, 'r') as file:
    lines = file.readlines()

# Step 1: Extract the lines where the regression results are
data_lines = []
start_reading = False

# Find the section where the results start (can be adjusted as needed)
for line in lines:
    if line.strip().startswith("const"):  # Look for the header or first result row
        start_reading = True
    if start_reading:
        data_lines.append(line.strip())

# Step 2: Parse the data from the lines (split by spaces or tabs)
parsed_data = []

for line in data_lines:
    # Split each line by whitespace and extract the data
    parts = re.split(r'\s+', line)  # This handles spaces and tabs
    if len(parts) == 7:  # Check if the line contains 7 columns (feature, coef, std_err, t, P>|t|, CI_025, CI_975)
        parsed_data.append(parts)

# Step 3: Convert the parsed data into a pandas DataFrame
columns = ['Feature', 'Coefficient', 'Std_Err', 't_Statistic', 'P_>|t|', 'CI_025', 'CI_975']
df = pd.DataFrame(parsed_data, columns=columns)

# Convert the numerical columns to the correct data types (coef, std_err, t_stat, etc.)
df['Coefficient'] = pd.to_numeric(df['Coefficient'], errors='coerce')
df['P_>|t|'] = pd.to_numeric(df['P_>|t|'], errors='coerce')

# Step 4: Filter significant features based on P-value <= 0.05
significant_neg = df.loc[(df['P_>|t|'] <= 0.05) & (df['Coefficient'] < 0)]['Feature'].tolist()
significant_pos = df.loc[(df['P_>|t|'] <= 0.05) & (df['Coefficient'] > 0)]['Feature'].tolist()

# Step 5: Print the results
print("Significant negative features (more Iran-like):", significant_neg)
print("Significant positive features (more UK-like):", significant_pos)

# Optional: Save the filtered results to a CSV
output_csv_path = "/home/shekhar/Desktop/code/persona/filtered_df_llama_8b_model_significant_features_LIWC.csv"
df.to_csv(output_csv_path, index=False)
print(f"Filtered features saved to {output_csv_path}")



[✅] LIWC-enriched data saved to: all_data_with_liwc.csv
                                 model culture                 topic  \
0  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
1  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
2  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
3  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
4  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
5  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
6  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
7  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
8  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   
9  meta-llama_Llama-3.2-1B-Instruct_IR    Iran  Interracial Marriage   

   temperature  generation_number       character_pair        stance  \
0          0.9                  4  Mother and Daughter       Neutral   
1     

/tmp/ipykernel_1877641/4125445931.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df1['culture'] = df1['culture'].replace({'UK': 1, 'Iran': 0})
/tmp/ipykernel_1877641/4125445931.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['culture'] = df1['culture'].replace({'UK': 1, 'Iran': 0})
/tmp/ipykernel_1877641/4125445931.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

                            OLS Regression Results                            
Dep. Variable:                culture   R-squared:                       0.781
Model:                            OLS   Adj. R-squared:                  0.779
Method:                 Least Squares   F-statistic:                     574.2
Date:                Thu, 06 Nov 2025   Prob (F-statistic):               0.00
Time:                        16:14:08   Log-Likelihood:                 398.51
No. Observations:               12000   AIC:                            -647.0
Df Residuals:                   11925   BIC:                            -92.57
Df Model:                          74                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 0.5000      0.00

### Importing necessary libraries

In [16]:
# os: Provides a way of interacting with the operating system, including file and directory manipulation
import os  
# json: Allows for working with JSON data, enabling easy reading and writing of JSON files
import json  
# pandas: A powerful data analysis library, used for handling data in table-like structures (DataFrame)
import pandas as pd  
# re: The regular expression module, used for pattern matching in strings
import re 
 # Used for creating plots and visualizations
import matplotlib.pyplot as plt 

### Read and Parse LIWC Dictionary

In [17]:
# Path to LIWC 2015 Dictionary file
liwc_file_path = "/home/shekhar/Desktop/code/persona/LIWC2015_Dictionary.dic"

# Initialize dictionaries and list to store results
category_mapping = {}
word_category_pairs = []

# Read the LIWC dictionary file
with open(liwc_file_path, 'r', encoding='utf-8') as file:
    is_category_section = False  # Initially set to False until the first '%' marker is encountered
    
    for line in file:
        line = line.strip()
        
        # Identify the start and end of the category section
        if line.startswith('%'):
            # Toggle the flag when encountering '%'
            is_category_section = not is_category_section
            continue  # Skip the line containing '%'

        # Process the category section (between the first and second '%')
        if is_category_section:
            parts = line.split(maxsplit=1)
            if len(parts) == 2:
                try:
                    category_number = int(parts[0])  # The number (e.g., 1)
                    category_label = parts[1]  # The label (e.g., "function")
                    category_mapping[category_number] = category_label  # Store in dictionary
                except ValueError:
                    continue  # Ignore any lines that can't be parsed correctly

        # Process the word-to-category section (after the second '%')
        else:
            parts = line.split()  # Split the line into words and category numbers
            if not parts:
                continue  # Skip empty lines

            word = parts[0]  # The word (e.g., "about")

            # Convert valid category numbers; skip non-numeric parts
            categories = []
            for cat in parts[1:]:
                if cat.isdigit():  # Check if the category part is numeric
                    categories.append(int(cat))  # Add to the list if it is a number

            # Append the word with its associated categories
            for category in categories:
                category_label = category_mapping.get(category, 'Unknown')  # Get category label
                word_category_pairs.append([word.lower(), category_label])

# Convert to a pandas DataFrame
df_liwc = pd.DataFrame(word_category_pairs, columns=['Word', 'Category'])
df_liwc

,Word,Category
0,(:,affect
1,(:,posemo
2,(:,informal
3,(:,netspeak
4,(;,informal
...,...,...
19349,zombie*,death
19350,zoom,relativ
19351,zoom,motion
19352,zz*,informal


### top 10 words Iran vs UK

In [19]:
from collections import Counter
import re
import pandas as pd

# ------------------------------
# 1️⃣  Build category_to_words from df_liwc
# ------------------------------
category_to_words = {}
for category in df_liwc['Category'].unique():
    words = set(df_liwc[df_liwc['Category'] == category]['Word'].str.lower())
    category_to_words[category] = words

print(f"[✅] LIWC category_to_words dictionary created with {len(category_to_words)} categories.")

# ------------------------------
# 2️⃣  Tokenizer
# ------------------------------
def tokenize_text(text):
    """Lowercase, remove non-alphabetic chars, split by space."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

# ------------------------------
# 3️⃣  Function to get top words per category
# ------------------------------
def top_words_for_category(texts, category_name, category_to_words, top_n=10):
    words_in_category = category_to_words.get(category_name, set())
    counter = Counter()
    for text in texts:
        tokens = tokenize_text(text)
        for token in tokens:
            if token in words_in_category:
                counter[token] += 1
    return counter.most_common(top_n)

# ------------------------------
# 4️⃣  Select top-10 categories for Iran and UK
# ------------------------------
top10_iran_categories = significant_neg[:10]
top10_uk_categories   = significant_pos[:10]

print("🟥 Top-10 Iran-like LIWC categories:", top10_iran_categories)
print("🟦 Top-10 UK-like LIWC categories:", top10_uk_categories)

# ------------------------------
# 5️⃣  Extract top-10 words for each category
# ------------------------------
results_iran = []
results_uk = []

for cat in top10_iran_categories:
    top_words = top_words_for_category(df_full['text'], cat, category_to_words, top_n=10)
    for word, freq in top_words:
        results_iran.append({'Category': cat, 'Word': word, 'Frequency': freq})

for cat in top10_uk_categories:
    top_words = top_words_for_category(df_full['text'], cat, category_to_words, top_n=10)
    for word, freq in top_words:
        results_uk.append({'Category': cat, 'Word': word, 'Frequency': freq})

# ------------------------------
# 6️⃣  Convert to DataFrame & save CSVs
# ------------------------------
df_top_iran = pd.DataFrame(results_iran)
df_top_uk = pd.DataFrame(results_uk)

iran_out = "/home/shekhar/Desktop/code/persona/top_words_Iran_like.csv"
uk_out   = "/home/shekhar/Desktop/code/persona/top_words_UK_like.csv"

df_top_iran.to_csv(iran_out, index=False)
df_top_uk.to_csv(uk_out, index=False)

print(f"\n✅ Saved Iran-like top words → {iran_out}")
print(f"✅ Saved UK-like top words → {uk_out}")

# ------------------------------
# 7️⃣  Optional preview
# ------------------------------
print("\n🟥 Iran-like top words (first few rows):")
print(df_top_iran.head(20))

print("\n🟦 UK-like top words (first few rows):")
print(df_top_uk.head(20))


[✅] LIWC category_to_words dictionary created with 73 categories.
🟥 Top-10 Iran-like LIWC categories: ['affect', 'netspeak', 'function', 'power', 'sad', 'body', 'discrep', 'health', 'sexual', 'assent']
🟦 Top-10 UK-like LIWC categories: ['const', 'posemo', 'informal', 'leisure', 'article', 'bio', 'achiev', 'adj', 'cogproc', 'prep']

✅ Saved Iran-like top words → /home/shekhar/Desktop/code/persona/top_words_Iran_like.csv
✅ Saved UK-like top words → /home/shekhar/Desktop/code/persona/top_words_UK_like.csv

🟥 Iran-like top words (first few rows):
    Category   Word  Frequency
0     affect     to     331226
1     affect   dear      80712
2     affect   love      80195
3     affect   good      55679
4     affect   well      52008
5     affect  sighs      20191
6     affect   glad      20013
7     affect   okay      18751
8     affect  agree      18510
9     affect  happy      16830
10  netspeak     eh       3935
11  netspeak     ha       2650
12  netspeak     ya        525
13  netspeak     

In [21]:
print(df_top_iran.head(100))

   Category        Word  Frequency
0    affect          to     331226
1    affect        dear      80712
2    affect        love      80195
3    affect        good      55679
4    affect        well      52008
..      ...         ...        ...
95   assent  absolutely       4974
96   assent      indeed       3813
97   assent          aw        204
98   assent        cool        152
99   assent          ok         43

[100 rows x 3 columns]
